In [ ]:
import os
import json
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
from skimage.draw import polygon
import matplotlib.pyplot as plt
from tqdm import tqdm
import gc
import warnings

warnings.filterwarnings('ignore')

# ==================== Device Setup ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.backends.cudnn.benchmark = True

# ==================== Hyperparameters ====================
num_epochs = 20
batch_size = 6
learning_rate = 2e-4
weight_decay = 1e-4
patience = 5
num_classes = 2
image_size = 256

# ==================== Dataset ====================
class CocoMaskedDataset(Dataset):
    def __init__(self, images_path, annotations_path, transform=None):
        self.images_path = images_path
        self.transform = transform
        with open(annotations_path, "r") as f:
            coco = json.load(f)
        self.images = {img["id"]: img["file_name"] for img in coco["images"]}
        self.image_ids = list(self.images.keys())
        self.annotations = {}
        for ann in coco["annotations"]:
            img_id = ann["image_id"]
            if img_id not in self.annotations:
                self.annotations[img_id] = []
            self.annotations[img_id].append(ann)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img = np.array(Image.open(os.path.join(self.images_path, self.images[img_id])).convert("RGB"))
        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        for ann in self.annotations.get(img_id, []):
            for seg in ann.get("segmentation", []):
                if isinstance(seg, list) and len(seg) % 2 == 0:
                    poly_pts = np.array(seg).reshape(-1, 2)
                    rr, cc = polygon(poly_pts[:, 1], poly_pts[:, 0], shape=img.shape[:2])
                    mask[rr, cc] = 1
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img = augmented["image"]
            mask = augmented["mask"]
        return img.float(), mask.long()

# ==================== Augmentations ====================
train_transform = A.Compose([
    A.LongestMaxSize(max_size=image_size),
    A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=0),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.3),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.LongestMaxSize(max_size=image_size),
    A.PadIfNeeded(min_height=image_size, min_width=image_size, border_mode=0),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

# ==================== Paths ====================
images_path = r"D:\val2017\val2017"
annotations_path = r"D:\annotations_trainval2017\annotations\instances_val2017.json"

# ==================== Dataset & DataLoader ====================
dataset = CocoMaskedDataset(images_path, annotations_path, transform=train_transform)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ==================== Model ====================
weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights, aux_loss=True)

# Replace classifier for our num_classes
old_cls = model.classifier
model.classifier = nn.Sequential(
    old_cls[0], old_cls[1], old_cls[2],
    nn.Dropout(0.3),
    nn.Conv2d(256, 128, kernel_size=3, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.Conv2d(128, num_classes, kernel_size=1)
)
model = model.to(device)

# ==================== Loss ====================
class DiceCELoss(nn.Module):
    def __init__(self, weight=None):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight)

    def dice_loss(self, pred, target):
        pred = torch.softmax(pred, dim=1)
        target_one_hot = torch.zeros_like(pred)
        target_one_hot.scatter_(1, target.unsqueeze(1), 1)
        intersection = (pred * target_one_hot).sum(dim=(2,3))
        union = pred.sum(dim=(2,3)) + target_one_hot.sum(dim=(2,3))
        dice = (2*intersection) / (union + 1e-7)
        return 1 - dice.mean()

    def forward(self, pred, target):
        return 0.5*self.ce(pred, target) + 0.5*self.dice_loss(pred, target)

criterion = DiceCELoss(weight=torch.tensor([0.3,0.7]).to(device))
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# ==================== Metrics ====================
def calculate_iou(preds, masks):
    preds = torch.argmax(preds, dim=1)
    intersection = ((preds==1) & (masks==1)).sum().float()
    union = ((preds==1) | (masks==1)).sum().float()
    return (intersection/(union+1e-7)).item()

def calculate_dice(preds, masks):
    preds = torch.argmax(preds, dim=1)
    intersection = ((preds==1) & (masks==1)).sum().float()*2
    total = (preds==1).sum().float() + (masks==1).sum().float()
    return (intersection/(total+1e-7)).item()

# ==================== Training ====================
best_iou = 0.0
patience_counter = 0

for epoch in range(1, num_epochs+1):
    print(f"\nEpoch {epoch}/{num_epochs}")

    # Train
    model.train()
    train_loss = 0
    for imgs, masks in tqdm(train_loader, desc="Training"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)['out']
        loss = criterion(outputs, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validate
    model.eval()
    val_loss, ious, dices = 0, [], []
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc="Validation"):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)['out']
            loss = criterion(outputs, masks)
            val_loss += loss.item()
            ious.append(calculate_iou(outputs, masks))
            dices.append(calculate_dice(outputs, masks))
    val_loss /= len(val_loader)
    mean_iou = np.mean(ious)
    mean_dice = np.mean(dices)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | IoU: {mean_iou:.4f} | Dice: {mean_dice:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    # Scheduler
    scheduler.step()

    # Save best
    if mean_iou > best_iou:
        best_iou = mean_iou
        torch.save(model.state_dict(), "deeplabv3_best.pth")
        print(f"✓ Best model saved (IoU: {best_iou:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    # Clear GPU cache
    if torch.cuda.is_available():
        torch.cuda

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights
import matplotlib.pyplot as plt
from skimage.draw import polygon
from tqdm import tqdm

# ================= Device =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================= Dataset =================
class CocoMaskedDataset(Dataset):
    def __init__(self, images_path, annotations_path, transform=None):
        with open(annotations_path, "r") as f:
            coco = json.load(f)
        self.images_path = images_path
        self.images = {img["id"]: img["file_name"] for img in coco["images"]}
        self.image_ids = list(self.images.keys())
        self.annotations = {}
        for ann in coco["annotations"]:
            img_id = ann["image_id"]
            if img_id not in self.annotations:
                self.annotations[img_id] = []
            self.annotations[img_id].append(ann)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img = np.array(Image.open(os.path.join(self.images_path, self.images[img_id])).convert("RGB"))
        mask = np.zeros(img.shape[:2], dtype=np.uint8)
        for ann in self.annotations.get(img_id, []):
            for seg in ann.get("segmentation", []):
                if isinstance(seg, list) and len(seg) % 2 == 0:
                    poly_pts = np.array(seg).reshape(-1, 2)
                    rr, cc = polygon(poly_pts[:, 1], poly_pts[:, 0], shape=img.shape[:2])
                    mask[rr, cc] = 1
        if self.transform:
            augmented = self.transform(image=img, mask=mask)
            img, mask = augmented["image"], augmented["mask"]
        return img.float(), mask.long()

# ================= Transforms =================
val_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
    ToTensorV2()
])

# ================= Paths =================
images_path = r"D:\val2017\val2017"
annotations_path = r"D:\annotations_trainval2017\annotations\instances_val2017.json"

val_dataset = CocoMaskedDataset(images_path, annotations_path, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

# ---- Subset validation set to 1000 images ----
val_subset = Subset(val_dataset, range(1000))
val_loader_small = DataLoader(val_subset, batch_size=4, shuffle=False)

# ================= Model =================
num_classes = 2
weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights, aux_loss=True)

# Modify classifier
old_cls = model.classifier
model.classifier = nn.Sequential(
    old_cls[0], old_cls[1], old_cls[2], nn.Dropout(0.3),
    nn.Conv2d(256, 128, kernel_size=3, padding=1),
    nn.BatchNorm2d(128),
    nn.ReLU(),
    nn.Conv2d(128, num_classes, kernel_size=1)
)
model = model.to(device)
model.load_state_dict(torch.load("deeplabv3_final.pth", map_location=device))
model.eval()
print("Model loaded")

# ================= Metrics =================
def compute_metrics(preds, masks, num_classes):
    ious, dices = [], []
    correct, total = 0, 0
    for pred, mask in zip(preds, masks):
        for cls in range(num_classes):
            pred_cls = (pred == cls)
            mask_cls = (mask == cls)
            intersection = (pred_cls & mask_cls).sum()
            union = (pred_cls | mask_cls).sum()
            if union > 0:
                ious.append(intersection / union)
            if (pred_cls.sum() + mask_cls.sum()) > 0:
                dices.append((2 * intersection) / (pred_cls.sum() + mask_cls.sum()))
        correct += (pred == mask).sum()
        total += mask.size
    return np.mean(ious), np.mean(dices), correct / total

# ================= Evaluation & Visualization =================
def evaluate_and_visualize(model, dataset, loader, num_classes, num_samples=5):
    model.eval()
    all_preds, all_masks = [], []

    # ---- Compute Metrics with progress bar ----
    print(f"Evaluating dataset ({len(loader.dataset)} images)...")
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Evaluation", unit="batch"):
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)['out']
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            masks_np = masks.cpu().numpy()
            all_preds.extend(preds)
            all_masks.extend(masks_np)

    mean_iou, mean_dice, acc = compute_metrics(all_preds, all_masks, num_classes)
    print(f"\nMetrics -> Mean IoU: {mean_iou:.4f}, Mean Dice: {mean_dice:.4f}, Accuracy: {acc:.4f}\n")

    # ---- Random Visualization ----
    print(f"Showing {num_samples} random samples...\n")
    indices = random.sample(range(len(dataset)), num_samples)

    for idx in indices:
        img, mask_gt = dataset[idx]
        img_in = img.unsqueeze(0).to(device)
        with torch.no_grad():
            pred = model(img_in)['out']
        pred = torch.argmax(pred, dim=1).squeeze().cpu().numpy()

        # Denormalize image
        img_vis = img.permute(1, 2, 0).cpu().numpy()
        img_vis = img_vis * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])
        img_vis = np.clip(img_vis, 0, 1)

        # Highlight only object
        highlighted_gt = img_vis * np.expand_dims(mask_gt.numpy(), axis=-1)
        highlighted_pred = img_vis * np.expand_dims(pred, axis=-1)

        # Plot
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1); plt.imshow(img_vis); plt.title("Image"); plt.axis("off")
        plt.subplot(1, 3, 2); plt.imshow(highlighted_gt); plt.title("Ground Truth"); plt.axis("off")
        plt.subplot(1, 3, 3); plt.imshow(highlighted_pred); plt.title("Prediction"); plt.axis("off")
        plt.show()

# ================= Run Evaluation on Subset =================
evaluate_and_visualize(model, val_dataset, val_loader_small, num_classes=num_classes, num_samples=5)


In [ ]:
import os
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from torchvision.models.segmentation import deeplabv3_resnet50
import torch.nn as nn

# ================= Device =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================= Model =================
num_classes = 2
model_path = "deeplabv3_final.pth"
model = deeplabv3_resnet50(weights=None, aux_loss=True)
old_cls = model.classifier
model.classifier = nn.Sequential(
    old_cls[0], old_cls[1], old_cls[2], nn.Dropout(0.3),
    nn.Conv2d(256, 128, kernel_size=3, padding=1),
    nn.BatchNorm2d(128), nn.ReLU(),
    nn.Conv2d(128, num_classes, kernel_size=1)
)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()
print("Model loaded")

# ================= Preprocess =================
def preprocess_image(img, long_side=256):
    h, w = img.shape[:2]
    scale = long_side / max(h, w)
    new_w, new_h = int(w*scale), int(h*scale)
    img_resized = cv2.resize(img, (new_w, new_h))
    img_tensor = torch.tensor(img_resized/255., dtype=torch.float32).permute(2,0,1).unsqueeze(0)
    mean = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
    std = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)
    img_tensor = (img_tensor - mean) / std
    return img_tensor.to(device), (h, w)

# ================= TTA Prediction =================
def tta_predict(img_tensor):
    aug_list = [
        lambda x: x,
        lambda x: torch.flip(x, [3]),
        lambda x: torch.flip(x, [2]),
        lambda x: torch.rot90(x, k=1, dims=[2,3]),
        lambda x: torch.rot90(x, k=3, dims=[2,3])
    ]
    outputs = []
    with torch.no_grad():
        for f in aug_list:
            t = f(img_tensor)
            out = model(t)['out']
            if f != aug_list[0]:
                if f == aug_list[1]: out = torch.flip(out,[3])
                elif f == aug_list[2]: out = torch.flip(out,[2])
                elif f == aug_list[3]: out = torch.rot90(out,k=3,dims=[2,3])
                elif f == aug_list[4]: out = torch.rot90(out,k=1,dims=[2,3])
            outputs.append(out)
    return torch.mean(torch.stack(outputs), dim=0)

# ================= Post-process mask (Smooth edges) =================
def postprocess_mask(mask_tensor, orig_size):
    mask = torch.argmax(mask_tensor, dim=1).squeeze().cpu().numpy()
    mask = cv2.resize(mask.astype(np.uint8), (orig_size[1], orig_size[0]), interpolation=cv2.INTER_NEAREST)
    
    # Morphological cleaning
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.dilate(mask, kernel, iterations=1)

    # Convert to 0-255
    mask = mask.astype(np.uint8) * 255

    # Gaussian blur for smooth edges
    blurred = cv2.GaussianBlur(mask, (5,5), 0)

    # Threshold back to binary (smooth outline)
    _, smooth_mask = cv2.threshold(blurred, 128, 255, cv2.THRESH_BINARY)

    return (smooth_mask // 255).astype(np.uint8)

# ================= Extract Object (Feathered edges) =================
def extract_object(img, mask):
    mask = mask.astype(np.float32)

    # Feather edges
    mask = cv2.GaussianBlur(mask, (7,7), 0)

    # Normalize to [0,1]
    mask = np.expand_dims(mask, axis=-1)
    mask = mask / (mask.max() + 1e-8)

    # Blend object with black background
    black_bg = np.zeros_like(img, dtype=np.float32)
    result = img.astype(np.float32) * mask + black_bg * (1 - mask)
    return result.astype(np.uint8)

# ================= Run on Folder =================
input_folder = r"D:\external images"  # Change to your folder
for filename in os.listdir(input_folder):
    if filename.lower().endswith((".jpg", ".jpeg", ".png")):
        path = os.path.join(input_folder, filename)
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Preprocess, TTA predict, post-process
        img_tensor, orig_size = preprocess_image(img)
        mask_tensor = tta_predict(img_tensor)
        pred_mask = postprocess_mask(mask_tensor, orig_size)
        # Extract object with smooth edges
        result = extract_object(img, pred_mask)
        # Display
        plt.figure(figsize=(12,5))
        plt.subplot(1,2,1); plt.imshow(img); plt.title("Original"); plt.axis("off")
        plt.subplot(1,2,2); plt.imshow(result); plt.title("Object (Smooth Edges)"); plt.axis("off")
        plt.show()
        print(f"Displayed: {filename}")
